# Tutorial 4 — Neural Networks as Nonlinear Ansatz Spaces

**MM845 — Tópicos de Geometria III: AI for Geometry**
Paired with **Lecture 4: Neural Networks — Theory, Architecture, Training**

---

Tutorial 3 fixed a feature map $\phi$ by hand and fitted a linear model on top of
it. Everything was provable: one minimum, a closed form, a convergence rate that
was a ratio of eigenvalues.

A neural network keeps the *shape* of that idea — still a variational problem over
a hypothesis space — and changes one thing: **$\phi$ is learned rather than
designed**. The price is the entire apparatus of guarantees. This tutorial measures
what the trade buys and what it costs, on the same kind of geometric target we have
been using throughout.

| § | Question |
|---|---|
| 1 | What is an MLP, as a space of functions? |
| 2 | Polynomials vs networks on targets of different regularity |
| 3 | The landscape is non-convex — how much does that matter? |
| 4 | Spectral bias: which part of a function is learned first |
| 5 | Summary |

You need the `aigeo` environment from [Tutorial 1](../tutorial_01/README.md); this
tutorial uses PyTorch throughout.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from itertools import combinations_with_replacement

SEED = 20260901
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

GEO_DARK, GEO_TEAL, GEO_RUST = "#103158", "#006c86", "#b2461e"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.prop_cycle": plt.cycler(color=[GEO_DARK, GEO_TEAL, GEO_RUST])})
print("torch", torch.__version__)

---
## 1. An MLP is a hypothesis space you cannot write down

A **multilayer perceptron** of depth $L$ alternates affine maps with a fixed
nonlinearity $\rho$:

$$f_\theta \;=\; A_L \circ \rho \circ A_{L-1} \circ \cdots \circ \rho \circ A_1,
\qquad A_i(u) = W_i u + b_i .$$

Two observations for a geometer, both important.

- $f_\theta$ is **linear in the last layer** and nonlinear in everything before it.
  So an MLP is exactly Tutorial 3's picture — a linear model on features
  $\phi_\theta(x)$ — except that $\phi_\theta$ moves during training. The last layer
  solves a least-squares problem; the rest of the network is busy choosing the basis
  it will solve it in.
- $\mathcal{H} = \{f_\theta : \theta \in \mathbb{R}^p\}$ is **not a vector space**.
  A sum of two networks is not a network of the same size. This is the single
  structural fact that destroys the theory: without linearity in $\theta$ there is
  no projection, no normal equations, and no reason for the loss to be convex.

What we get in exchange is expressivity. The universal approximation theorem says
$\mathcal{H}$ is dense in $C(K)$ for compact $K$ — but density is cheap, and says
nothing about how many parameters or how much data are needed. Those are the
questions worth asking experimentally.

In [ ]:
def make_mlp(d_in, width, depth=3, act=torch.nn.Tanh):
    """An MLP with `depth` hidden layers, all of the same width."""
    layers, d = [], d_in
    for _ in range(depth):
        layers += [torch.nn.Linear(d, width), act()]
        d = width
    layers += [torch.nn.Linear(d, 1)]
    return torch.nn.Sequential(*layers).double()


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def train(model, X, y, epochs=1500, lr=3e-3, verbose=False):
    """Lecture 3's canonical loop, unchanged."""
    Xt = torch.tensor(X); yt = torch.tensor(y).unsqueeze(1)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = torch.nn.MSELoss()
    hist = []
    for ep in range(epochs):
        opt.zero_grad()
        loss = lossf(model(Xt), yt)
        loss.backward()
        opt.step()
        hist.append(float(loss.detach()))
    return np.array(hist)


def predict(model, X):
    with torch.no_grad():
        return model(torch.tensor(X)).numpy().ravel()


m = make_mlp(3, 32)
print(m)
print(f"\nparameters: {n_params(m)}")

---
## 2. Polynomials versus networks

The comparison only means something if it is *fair*, so we match the two families by
**parameter count** and give both the same data and the same test set.

Three targets on $S^2$, chosen to span the regularity range that §4 of Tutorial 3
showed governs everything:

| target | regularity | in the polynomial span? |
|---|---|---|
| $g_1$: a degree-3 harmonic | analytic | **yes**, exactly |
| $g_2$: a narrow Gaussian bump | analytic, but sharply peaked | no |
| $g_3$: $\lvert z \rvert$ | Lipschitz, not $C^1$ | no |

In [ ]:
def sample_sphere(n, rng):
    x = rng.normal(size=(n, 3))
    return x / np.linalg.norm(x, axis=1, keepdims=True)


def fibonacci_sphere(n):
    i = np.arange(n); z = 1 - 2 * (i + 0.5) / n
    r = np.sqrt(np.maximum(0.0, 1 - z**2))
    th = 2 * np.pi * i / ((1 + 5**0.5) / 2)
    return np.stack([r * np.cos(th), r * np.sin(th), z], axis=1)


BUMP = np.array([0.4, 0.5, 0.76]); BUMP /= np.linalg.norm(BUMP)

def g_harmonic(X):
    x, y, z = X.T
    return 0.6 * (2 * z**2 - x**2 - y**2) + 1.1 * x * y + 0.9 * (x**3 - 3 * x * y**2)

def g_bump(X, w=0.28):
    return np.exp(-(2 - 2 * (X @ BUMP)) / (2 * w**2))

def g_kink(X):
    return np.abs(X[:, 2])


TARGETS = [("degree-3 harmonic", g_harmonic), ("Gaussian bump", g_bump), ("$|z|$", g_kink)]

N_TRAIN = 1200
X_tr = sample_sphere(N_TRAIN, rng)
X_te = fibonacci_sphere(20000)
print(f"{N_TRAIN} training points, {len(X_te)} test points (noiseless)")

In [ ]:
def poly_features(X, degree):
    cols = [np.ones(len(X))]
    for d in range(1, degree + 1):
        for c in combinations_with_replacement(range(X.shape[1]), d):
            cols.append(np.prod(X[:, c], axis=1))
    return np.stack(cols, axis=1)


def poly_fit_mse(X_tr, y_tr, X_te, y_te, degree):
    F = poly_features(X_tr, degree)
    w, *_ = np.linalg.lstsq(F, y_tr, rcond=None)
    return F.shape[1], np.mean((poly_features(X_te, degree) @ w - y_te)**2)


DEGREES = [2, 4, 6, 8, 10, 12]
WIDTHS = [4, 8, 16, 32, 64]
results = {}

for name, g in TARGETS:
    y_tr, y_te = g(X_tr), g(X_te)
    poly = [poly_fit_mse(X_tr, y_tr, X_te, y_te, D) for D in DEGREES]
    net = []
    for w in WIDTHS:
        torch.manual_seed(0)
        model = make_mlp(3, w, depth=3)
        train(model, X_tr, y_tr, epochs=1500)
        net.append((n_params(model), np.mean((predict(model, X_te) - y_te)**2)))
    results[name] = (np.array(poly), np.array(net))
    print(f"{name:20s} best poly {poly[int(np.argmin([p[1] for p in poly]))][1]:.2e}   "
          f"best MLP {min(n[1] for n in net):.2e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.2), sharey=True)
for ax, (name, _) in zip(axes, TARGETS):
    poly, net = results[name]
    ax.loglog(poly[:, 0], poly[:, 1], "o-", color=GEO_TEAL, label="polynomial")
    ax.loglog(net[:, 0], net[:, 1], "s-", color=GEO_RUST, label="MLP (depth 3)")
    ax.set_xlabel("number of parameters"); ax.set_title(name)
axes[0].set_ylabel("test MSE"); axes[0].legend(fontsize=8)
fig.suptitle("same data, same budget: which ansatz space is better?", y=1.03)
plt.tight_layout(); plt.show()

The ordering flips with regularity, exactly as it should.

- On the **harmonic** target the polynomial basis is unbeatable: the truth lies in
  the span, so least squares finds it to machine precision with 20 parameters. The
  network spends thousands of parameters approximating something it could never
  represent exactly, and stops several orders of magnitude short. *When your ansatz
  contains the answer, use it.*
- On the **bump** the polynomial still wins, and by a wide margin — a couple of
  hundred times lower error at comparable cost. The function is analytic, so its
  coefficients decay geometrically (Tutorial 3 §4) and a moderate degree captures it.
  Meanwhile the network is stuck at the level its optimiser happened to reach.
- Only on the **kink** does the network win. $|z|$ is exactly the kind of function
  polynomials handle badly — Gibbs-type oscillation near the crease — and that
  piecewise-smooth architectures handle naturally.

That is a more sobering result than the usual story, and it is worth stating
plainly: **on smooth targets in low dimension, classical approximation beats a
neural network comfortably.** Networks are not a better hammer for these problems.
What they offer is that they keep working when the target is rough, when the input
dimension is large enough that a polynomial basis becomes combinatorially
impossible, or when no good basis is known — which is precisely the regime the rest
of this course lives in.

The moral is not "networks are better". It is that Lecture 2's bias–variance
picture is a statement about *the fit between hypothesis space and truth*, and that
fit is a regularity question you can often answer before training anything.

> **Exercise 1 — where is the crossover?**
> (a) Interpolate between $g_1$ and $g_3$ by fitting $g_\alpha = (1-\alpha)g_1 +
> \alpha g_3$ for $\alpha \in [0,1]$, and find the $\alpha$ at which the MLP
> overtakes the polynomial at fixed parameter count.
>
> (b) Repeat the kink experiment with `act=torch.nn.ReLU`. A ReLU network is
> piecewise linear; does that help or hurt on $|z|$, and on the smooth bump?
>
> (c) Sweep $N_{\text{train}}$ and plot test MSE against $N$ for both families on
> the bump. Which has the steeper learning curve, and what does that say about
> which to reach for when data is expensive?

---
## 3. The landscape is non-convex — how much does that cost?

Lecture 3 could promise a unique minimiser. Lecture 4 cannot: $L(\theta)$ for a
network has saddles, valleys, and vast families of exactly equivalent parameters
(permute two hidden units and the function is unchanged).

That sounds alarming. The experiment below asks how alarming it actually is: train
the *same architecture* on the *same data* from eight different random
initialisations and see how much the outcome varies.

In [ ]:
y_tr, y_te = g_bump(X_tr), g_bump(X_te)
runs = []
for seed in range(8):
    torch.manual_seed(seed)
    model = make_mlp(3, 32, depth=3)
    hist = train(model, X_tr, y_tr, epochs=1500)
    runs.append((hist, predict(model, X_te)))

final_train = np.array([h[-1] for h, _ in runs])
final_test = np.array([np.mean((p - y_te)**2) for _, p in runs])
preds = np.stack([p for _, p in runs])

print(f"train loss  min {final_train.min():.2e}   max {final_train.max():.2e}"
      f"   spread x{final_train.max()/final_train.min():.1f}")
print(f"test  MSE   min {final_test.min():.2e}   max {final_test.max():.2e}"
      f"   spread x{final_test.max()/final_test.min():.1f}")
print(f"pointwise spread between runs: mean std {preds.std(axis=0).mean():.4f}"
      f"   (target range {np.ptp(y_te):.2f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.2))

for h, _ in runs:
    axes[0].loglog(h, lw=1, alpha=0.8)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("training loss")
axes[0].set_title("8 seeds, same architecture and data")

axes[1].scatter(final_train, final_test, s=40, color=GEO_DARK, zorder=3)
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("final training loss"); axes[1].set_ylabel("test MSE")
axes[1].set_title("different minima, comparable quality")

order = np.argsort(y_te)
axes[2].plot(y_te[order], preds.std(axis=0)[order], ".", ms=1.5, color=GEO_RUST)
axes[2].set_xlabel("target value"); axes[2].set_ylabel("std across seeds")
axes[2].set_title("where the runs disagree")
plt.tight_layout(); plt.show()

The runs land in *different* minima — the parameters are not close, and the
functions differ pointwise — yet the test errors agree to within a small factor.
This is the empirical fact that makes deep learning workable despite the missing
theory: in these landscapes, most minima that a gradient method actually reaches
are about equally good.

Two practical consequences, both of which Lecture 4 flagged and both of which you
should carry into your project:

- **Report a spread, not a number.** A single run's test error is a sample from a
  distribution. Quoting it alone is the computational equivalent of quoting one
  measurement without an error bar.
- **Fix and record the seed** (Lecture 1), so a run can be reproduced exactly — and
  then vary it deliberately, so you know how much of your result is the seed.

Notice also *where* the seeds disagree: predominantly in the region where the
target varies fastest. Disagreement between runs is a usable, if crude, proxy for
where a model is unreliable.

> **Exercise 2 — is it the seed, or the landscape?**
> (a) Re-run with a much wider network (width 128). Does the seed-to-seed spread
> grow or shrink? Relate your answer to the double-descent discussion of Lecture 2.
>
> (b) Keep the initialisation fixed and vary only the order in which minibatches are
> drawn (use `torch.optim.SGD` with shuffling). How much of the variability is
> initialisation and how much is SGD noise?
>
> (c) Average the predictions of the eight runs and measure the test error of that
> ensemble. Compare with the best individual run.

---
## 4. Spectral bias: networks learn low frequencies first

Tutorial 3 showed that for a *fixed* basis the error is governed by coefficient
decay. Networks have no fixed basis — but they have a strong, reproducible
preference about which frequencies they fit first, and it is one of the most useful
things to know about them.

We watch it on $S^1$, where the Fourier decomposition is one line of NumPy. Take a
target with one slow and one fast component,

$$f(\theta) \;=\; \sin\theta \;+\; \tfrac{1}{2}\sin 9\theta ,$$

train an MLP on $(\cos\theta, \sin\theta)$, and record the error in each Fourier
mode as training proceeds.

In [ ]:
n_th = 512
theta = np.linspace(0, 2 * np.pi, n_th, endpoint=False)
X_circ = np.stack([np.cos(theta), np.sin(theta)], axis=1)
f_circ = np.sin(theta) + 0.5 * np.sin(9 * theta)

torch.manual_seed(3)
net = make_mlp(2, 64, depth=3)
opt = torch.optim.Adam(net.parameters(), lr=3e-3)
lossf = torch.nn.MSELoss()
Xt = torch.tensor(X_circ); yt = torch.tensor(f_circ).unsqueeze(1)

SNAPS = [0, 50, 150, 400, 1200, 4000]
snapshots, amp_hist, epochs_rec = {}, [], []
for ep in range(4001):
    if ep in SNAPS:
        snapshots[ep] = predict(net, X_circ)
    if ep % 25 == 0:
        resid = predict(net, X_circ) - f_circ
        amp_hist.append(np.abs(np.fft.rfft(resid)) / n_th)
        epochs_rec.append(ep)
    opt.zero_grad(); lossf(net(Xt), yt).backward(); opt.step()

amp_hist = np.array(amp_hist)
print(f"recorded {len(epochs_rec)} spectra up to epoch {epochs_rec[-1]}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.4, 3.3))

for ep, col in zip(SNAPS, plt.cm.viridis(np.linspace(0, 0.9, len(SNAPS)))):
    axes[0].plot(theta, snapshots[ep], color=col, lw=1.3, label=f"epoch {ep}")
axes[0].plot(theta, f_circ, "k--", lw=1.2, label="target")
axes[0].set_xlabel(r"$\theta$"); axes[0].set_title("the fit, as training proceeds")
axes[0].legend(fontsize=6, ncol=2)

for k, col in zip([1, 9], [GEO_TEAL, GEO_RUST]):
    axes[1].loglog(np.array(epochs_rec) + 1, amp_hist[:, k], color=col, lw=1.8,
                   label=f"mode $k = {k}$")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("residual amplitude")
axes[1].set_title("the slow mode is fitted first"); axes[1].legend(fontsize=8)

im = axes[2].imshow(np.log10(amp_hist[:, :16].T + 1e-6), aspect="auto", origin="lower",
                    cmap="magma", extent=[0, epochs_rec[-1], 0, 15])
axes[2].set_xlabel("epoch"); axes[2].set_ylabel("Fourier mode $k$")
axes[2].set_title("residual spectrum, $\\log_{10}$")
fig.colorbar(im, ax=axes[2], shrink=0.85)
plt.tight_layout(); plt.show()

i1 = next(i for i, a in enumerate(amp_hist[:, 1]) if a < 0.05)
i9 = next((i for i, a in enumerate(amp_hist[:, 9]) if a < 0.05), len(epochs_rec) - 1)
print(f"epochs to bring mode 1 below 0.05: {epochs_rec[i1]}")
print(f"epochs to bring mode 9 below 0.05: {epochs_rec[i9]}")

The network fits $\sin\theta$ almost immediately and takes an order of magnitude
longer to reach $\sin 9\theta$. The middle panel makes it quantitative; the right
panel shows the whole residual spectrum draining from the bottom up.

This is **spectral bias** (sometimes the *frequency principle*), and it is worth
knowing for three separate reasons:

- It is an **implicit regulariser**. Early stopping on a network is approximately
  a low-pass filter — which is to say, approximately the Dirichlet-energy penalty of
  Tutorial 3 §4, arrived at by a completely different route.
- It explains why networks generalise better than their parameter count suggests:
  left to itself, gradient descent prefers the smooth explanation.
- It is a **problem** for the applications at the end of this course. A PINN
  solving a PDE with fine-scale structure (Lecture 13) is fighting this bias, and
  stiffness in the loss is exactly the symptom.

> **Exercise 3 — bias, measured.**
> (a) Fit $\sum_{k \in \{1,3,9,17\}} \sin k\theta$ and plot the epoch at which each
> mode's residual crosses a threshold, against $k$. Is the relationship a power law?
>
> (b) Replace the input $(\cos\theta,\sin\theta)$ by a **Fourier feature** embedding
> $(\cos\theta, \sin\theta, \cos 8\theta, \sin 8\theta, \dots)$ and repeat. This is
> the standard fix, and it should largely remove the bias — explain why in terms of
> Tutorial 3's feature maps.
>
> (c) Does the bias depend on the activation? Compare `Tanh` with `ReLU` and with
> `torch.nn.SiLU`.

---
## 5. What to take away

- An MLP is **a linear model on a learned feature map**. The last layer does
  Tutorial 3's least squares; everything before it chooses the basis.
- The hypothesis space is **not a vector space**, and that single fact removes the
  projection theorem, the closed form, and convexity all at once.
- **Regularity decides who wins.** If the truth lies in your polynomial span, use
  the polynomial: it is exact, cheap and interpretable. Networks earn their keep on
  targets that no convenient fixed basis represents well.
- **Non-convexity is survivable but not free.** Different seeds reach different
  minima of comparable quality, so results are distributions — report spreads, fix
  seeds, and treat between-run disagreement as a rough uncertainty map.
- **Networks learn smooth structure first.** Spectral bias is a free regulariser
  when you want smoothness and an obstacle when you do not; both cases recur later
  in the course.

### Next

**Lecture 5** constrains the architecture instead of enlarging it: convolutions are
linear maps that commute with translation. **Tutorial 5** builds a CNN on geometric
image data and measures what that symmetry constraint is worth.

### Further reading

- Rahaman et al., "On the spectral bias of neural networks", *ICML* 2019 — §4 in full.
- Tancik et al., "Fourier features let networks learn high frequency functions in low dimensional domains", *NeurIPS* 2020 — the fix in Exercise 3(b).
- Goodfellow, Bengio & Courville, *Deep Learning*, ch. 6 & 8 — MLPs and the optimisation landscape. Free online.
- Trefethen, *Approximation Theory and Approximation Practice*, ch. 9 — why polynomials struggle with $|x|$.